# **Fine-Tune Pre-Trained Classification Model With QLORA**

### **Import Libraries**

In [3]:
!pip install evaluate datasets transformers peft bitsandbytes accelerate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.2 MB/s eta 0:00:00


In [4]:
import torch
import numpy as np
import torch.nn as nn
import json
import evaluate
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training, LoraConfig, TaskType,get_peft_model, replace_lora_weights_loftq
from transformers import TrainingArguments, Trainer

In [5]:
device = torch.device("cuda" if torch.cuda.is_available else "cpu")
device.type

'cuda'

In [6]:
def save_data_to_file (data, file_path):

    with open(file_path, "w") as file:
        json.dump(data, file,indent=4)
    print(f"file created successfully at {file_path}")

In [7]:
def load_data_from_file(file_path):

    with open(file_path, "r") as file:
        load_file = json.load(file)

    return load_file

### **Load Dataset**

In [8]:
dataset = load_dataset("imdb")
dataset

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [9]:
train_labels = dataset['train']['label']
num_classes = len(set(train_labels))
num_classes

2

In [10]:
imdb_labels = {0: 'negative review', 1: 'positive review'}

In [11]:
# load tokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

##### Tokenizer Map Function

In [12]:
def tokenized_text (data):
    return tokenizer(data['text'],
                     padding='max_length', max_length=512, truncation=True)

In [13]:
dataset = dataset.map(tokenized_text, batched=True)

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [14]:
dataset['train'][0]

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

In [15]:
dataset = dataset.rename_column('label', 'labels')

In [16]:
dataset.set_format(type='torch', columns = ['labels', 'input_ids', 'attention_mask'])

In [17]:
dataset['train'][0]

{'labels': tensor(0),
 'input_ids': tensor([  101,  1045, 12524,  1045,  2572,  8025,  1011,  3756,  2013,  2026,
          2678,  3573,  2138,  1997,  2035,  1996,  6704,  2008,  5129,  2009,
          2043,  2009,  2001,  2034,  2207,  1999,  3476,  1012,  1045,  2036,
          2657,  2008,  2012,  2034,  2009,  2001,  8243,  2011,  1057,  1012,
          1055,  1012,  8205,  2065,  2009,  2412,  2699,  2000,  4607,  2023,
          2406,  1010,  3568,  2108,  1037,  5470,  1997,  3152,  2641,  1000,
          6801,  1000,  1045,  2428,  2018,  2000,  2156,  2023,  2005,  2870,
          1012,  1026,  7987,  1013,  1028,  1026,  7987,  1013,  1028,  1996,
          5436,  2003,  8857,  2105,  1037,  2402,  4467,  3689,  3076,  2315,
         14229,  2040,  4122,  2000,  4553,  2673,  2016,  2064,  2055,  2166,
          1012,  1999,  3327,  2016,  4122,  2000,  3579,  2014,  3086,  2015,
          2000,  2437,  2070,  4066,  1997,  4516,  2006,  2054,  1996,  2779,
         25430, 1

In [18]:
dataset_train = dataset['train']
dataset_test = dataset['test']

### **BitsAndBytes Configuration**

In [19]:
config_bits = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype= torch.bfloat16,
    llm_int8_skip_modules=['classifier', 'pre_classifier']
)

In [20]:
#ids to labels
id_to_label = {0: "Negative", 1: "Positive"}

# labels to ids
label_to_id = dict((v,k) for k,v in id_to_label.items())

### **Define Model Inatance**

In [21]:
qlora_model = AutoModelForSequenceClassification.from_pretrained(
                                                "distilbert-base-uncased",
                                                id2label = id_to_label,
                                                label2id = label_to_id,
                                                num_labels = num_classes,
                                                quantization_config = config_bits
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


After loading the model with 4-bit or 8-bit quantization, the model is only quantized for inference/memory efficiency. It is not yet configured properly for stable training.

This step prepares it for QLoRA fine-tuning. Internally, PEFT modifies the model for low-bit training by:

* freezing the original pretrained weights
* enabling gradients only where needed
* converting sensitive layers (like layer norms) to higher precision (float32)
* improving numerical stability during backpropagation
* optionally enabling gradient checkpointing support

In [22]:
qlora_model = prepare_model_for_kbit_training(qlora_model)

#### **Define PERF Model**

In [23]:
# define lora config
lora_config = LoraConfig(
    task_type= TaskType.SEQ_CLS,
    r = 8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=['q_lin', 'k_lin', 'v_lin']
)

peft_model = get_peft_model(qlora_model, lora_config)

peft_model_qlora is now configured as a QLoRA model and ready for training. Before starting the training process, an additional optimization step will be applied by reinitializing the LoRA weights with LoftQ, a technique that has demonstrated improved performance for quantized model training.

In [24]:
replace_lora_weights_loftq(peft_model)

In [25]:
print(peft_model)

PeftModelForSequenceClassification(
  (base_model): LoraModel(
    (model): DistilBertForSequenceClassification(
      (distilbert): DistilBertModel(
        (embeddings): Embeddings(
          (word_embeddings): Embedding(30522, 768, padding_idx=0)
          (position_embeddings): Embedding(512, 768)
          (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (transformer): Transformer(
          (layer): ModuleList(
            (0-5): 6 x TransformerBlock(
              (attention): DistilBertSelfAttention(
                (q_lin): lora.Linear4bit(
                  (base_layer): Linear4bit(in_features=768, out_features=768, bias=True)
                  (lora_dropout): ModuleDict(
                    (default): Dropout(p=0.1, inplace=False)
                  )
                  (lora_A): ModuleDict(
                    (default): Linear(in_features=768, out_features=8, bias=False)
                  

In [26]:
peft_model.print_trainable_parameters()

trainable params: 813,314 || all params: 67,768,324 || trainable%: 1.2001


### **Define Model Training**

In [31]:
training_args = TrainingArguments(
    "./result_qlora",
    num_train_epochs=10,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    weight_decay=0.1,
    logging_strategy="steps",
    logging_steps=100,
    fp16=True
)

In [32]:
def compute_accuracy (predict_eval):

    load_accuracy = evaluate.load("accuracy")

    logits, labels = predict_eval

    prediction = np.argmax(logits, axis=-1)
    accuracy = load_accuracy.compute(predictions=prediction, references=labels)['accuracy']

    return {"accuracy": accuracy}

In [33]:
trainer = Trainer(
    model= peft_model,
    args=training_args,
    train_dataset=dataset_train,
    eval_dataset=dataset_test,
    processing_class=tokenizer,
    compute_metrics=compute_accuracy
)

In [34]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss,Accuracy
1,0.290665,0.267068,0.893920
2,0.285097,0.251952,0.904640
3,0.241343,0.236863,0.910880
4,0.215679,0.235861,0.913040
5,0.206184,0.243772,0.914440
6,0.225369,0.234522,0.915160
7,0.202837,0.237661,0.916280
8,0.252105,0.234882,0.917760
9,0.222802,0.233157,0.917400
10,0.267855,0.235620,0.917680


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt

TrainOutput(global_step=31250, training_loss=0.25109587872314454, metrics={'train_runtime': 7798.0627, 'train_samples_per_second': 32.059, 'train_steps_per_second': 4.007, 'total_flos': 3.3741474816e+16, 'train_loss': 0.25109587872314454, 'epoch': 10.0})

In [38]:
torch.save(peft_model.state_dict(), "peft_qlora_imdb_model.pt")

fatal: not a git repository (or any of the parent directories): .git


In [66]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [41]:
import os

In [42]:

# clone your existing repo with the specific branch
!git clone -b qlora-finetune https://github.com/VishSeran/fine-tune-classification-model-with-adapters.git /content/drive/MyDrive/fine-tune-classification-model-with-adapters


Cloning into '/content/drive/MyDrive/fine-tune-classification-model-with-adapters'...
remote: Enumerating objects: 369, done.
remote: Counting objects: 100% (143/143), done.
remote: Compressing objects: 100% (132/132), done.
remote: Total 369 (delta 21), reused 46 (delta 10), pack-reused 226 (from 1)
Receiving objects: 100% (369/369), 3.58 MiB | 8.13 MiB/s, done.
Resolving deltas: 100% (180/180), done.


In [43]:
os.chdir("/content/drive/MyDrive/fine-tune-classification-model-with-adapters")

In [45]:
!git branch
!git log --oneline -5

* qlora-finetune
88e1c0e (HEAD -> qlora-finetune, origin/qlora-finetune) model training updated
88dd231 model trainer updated
db4f30d accuracy function updated
8c9aafb model trainer updated
a0afefa bits and bytes configured


In [67]:
!ls /content/drive/MyDrive/fine-tune-classification-model-with-adapters/fine-tune-wth-QLORA

QLORA-classification-pretrained-model.ipynb	     result_qlora
QLORA-classification-pretrained-model_updated.ipynb
